# E-commerce Fulfilment & Delivery Performance Analysis

## Notebook 02: Data Cleaning and Preparation

This notebook focuses on cleaning and preparing the raw Olist e-commerce data for analysis.

The purpose of this notebook is to:

- Load the raw CSV files
- Convert important date columns into proper datetime format
- Focus on delivered orders for delivery performance analysis
- Create delivery-related calculated fields
- Prepare clean datasets for later SQL, Power BI and exploratory analysis

The first notebook focused on understanding the raw data. This notebook starts preparing the data for analysis.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Define project folders

raw_data_path = Path("../data/raw")
processed_data_path = Path("../data/processed")

# Check that both folders exist
print("Raw data folder exists:", raw_data_path.exists())
print("Processed data folder exists:", processed_data_path.exists())

Raw data folder exists: True
Processed data folder exists: True


## Load Raw Data

In this section, I will load the raw CSV files again.

Although the files were already loaded in Notebook 01, each notebook should be able to run independently. This means Notebook 02 should not depend on Notebook 01 being open or already run.

In [3]:
# Load raw datasets

customers = pd.read_csv(raw_data_path / "olist_customers_dataset.csv")
order_items = pd.read_csv(raw_data_path / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(raw_data_path / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(raw_data_path / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(raw_data_path / "olist_orders_dataset.csv")
products = pd.read_csv(raw_data_path / "olist_products_dataset.csv")
sellers = pd.read_csv(raw_data_path / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(raw_data_path / "product_category_name_translation.csv")

print("Raw datasets loaded successfully.")

Raw datasets loaded successfully.


## Initial Orders Table Check

The orders table is the main table for delivery performance analysis because it contains:

- Order ID
- Customer ID
- Order status
- Purchase timestamp
- Carrier delivery date
- Customer delivery date
- Estimated delivery date

Before cleaning, I will inspect the structure of the orders table.

In [4]:
# Preview the orders table

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
# Check columns and data types in the orders table

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


## Convert Date Columns

The raw orders table contains several date columns.

At first, pandas reads these columns as text. To calculate delivery days and late delivery, these columns need to be converted into datetime format.

Datetime format allows Python to calculate the difference between dates.

In [6]:
# Create a copy of the orders table for cleaning

orders_clean = orders.copy()

# List of date columns to convert
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert date columns to datetime format
for column in date_columns:
    orders_clean[column] = pd.to_datetime(orders_clean[column], errors="coerce")

# Check data types after conversion
orders_clean[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## Filter to Delivered Orders

For delivery performance analysis, I will focus on orders with an order status of delivered.

This is because late delivery can only be measured properly when the actual customer delivery date is available.

Cancelled, unavailable, shipped or invoiced orders may be useful for other analysis later, but they are not the main focus for delivery performance KPIs.

In [7]:
# Check order status counts

orders_clean["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [8]:
# Filter to delivered orders only

delivered_orders = orders_clean[orders_clean["order_status"] == "delivered"].copy()

# Check the number of delivered orders
print("Total orders:", len(orders_clean))
print("Delivered orders:", len(delivered_orders))

Total orders: 99441
Delivered orders: 96478


## Create Delivery Performance Fields

In this section, I will create new calculated fields for delivery analysis.

These fields will help answer key business questions such as:

- How long did orders take to reach customers?
- Were orders delivered before or after the estimated delivery date?
- How many days late were late orders?
- Which orders were delivered late?

In [9]:
# Create delivery performance fields

delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"] 
    - delivered_orders["order_purchase_timestamp"]
).dt.days

delivered_orders["estimated_delivery_days"] = (
    delivered_orders["order_estimated_delivery_date"] 
    - delivered_orders["order_purchase_timestamp"]
).dt.days

delivered_orders["delay_days"] = (
    delivered_orders["order_delivered_customer_date"] 
    - delivered_orders["order_estimated_delivery_date"]
).dt.days

delivered_orders["is_late_delivery"] = delivered_orders["delay_days"] > 0

# Preview the new fields
delivered_orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "estimated_delivery_days",
        "delay_days",
        "is_late_delivery"
    ]
].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,delay_days,is_late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.0,15,-8.0,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.0,19,-6.0,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.0,26,-18.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.0,26,-13.0,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.0,12,-10.0,False


In [10]:
# Summary statistics for delivery performance fields

delivered_orders[
    [
        "delivery_days",
        "estimated_delivery_days",
        "delay_days"
    ]
].describe()

,delivery_days,estimated_delivery_days,delay_days
count,96470.000000,96478.000000,96470.000000
mean,12.093604,23.372759,-11.875889
std,9.551380,8.758137,10.182105
min,0.000000,2.000000,-147.000000
25%,6.000000,18.000000,-17.000000
50%,10.000000,23.000000,-12.000000
75%,15.000000,28.000000,-7.000000
max,209.000000,155.000000,188.000000


In [11]:
# Late delivery count and rate

late_delivery_count = delivered_orders["is_late_delivery"].sum()
delivered_order_count = len(delivered_orders)
late_delivery_rate = late_delivery_count / delivered_order_count

print("Delivered orders:", delivered_order_count)
print("Late deliveries:", late_delivery_count)
print("Late delivery rate:", round(late_delivery_rate * 100, 2), "%")

Delivered orders: 96478
Late deliveries: 6534
Late delivery rate: 6.77 %


## Export Cleaned Orders Dataset

The cleaned delivered orders dataset will be saved into the processed data folder.

This processed file will be used later for SQL analysis, Power BI dashboards and further Python analysis.

In [12]:
# Export cleaned delivered orders dataset

output_file = processed_data_path / "cleaned_delivered_orders.csv"

delivered_orders.to_csv(output_file, index=False)

print("Cleaned delivered orders file saved to:")
print(output_file)

Cleaned delivered orders file saved to:
..\data\processed\cleaned_delivered_orders.csv


In [13]:
# Check that the processed file was created

output_file.exists()

True

## Create Analysis-Ready Master Dataset

The cleaned delivered orders table contains the delivery performance fields, but most business analysis requires data from multiple tables.

In this section, I will create one analysis-ready dataset by combining:

- delivered orders
- customer location
- order item and freight information
- payment information
- review score
- product category information
- seller location information

Some tables contain more than one row per order, so I will aggregate them to order level before joining. This avoids accidentally duplicating orders during the merge.

In [14]:
# Prepare customer location data

customers_clean = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ]
].copy()

customers_clean.head()

,customer_id,customer_unique_id,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,campinas,SP


In [15]:
# Aggregate order item data to order level

order_items_summary = (
    order_items
    .groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        total_item_value=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        avg_item_price=("price", "mean")
    )
    .reset_index()
)

order_items_summary.head()

,order_id,order_item_count,product_count,seller_count,total_item_value,total_freight_value,avg_item_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,239.90
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,199.90


In [16]:
# Aggregate payment data to order level

payments_summary = (
    order_payments
    .groupby("order_id")
    .agg(
        payment_count=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        max_payment_installments=("payment_installments", "max")
    )
    .reset_index()
)

payments_summary.head()

,order_id,payment_count,total_payment_value,max_payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3


In [17]:
# Aggregate review data to order level

reviews_summary = (
    order_reviews
    .groupby("order_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        review_count=("review_id", "count")
    )
    .reset_index()
)

reviews_summary.head()

,order_id,avg_review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [18]:
# Join order items to product and seller details

order_items_enriched = (
    order_items
    .merge(products, on="product_id", how="left")
    .merge(category_translation, on="product_category_name", how="left")
    .merge(sellers, on="seller_id", how="left")
)

order_items_enriched.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,87900,loanda,PR


In [19]:
# Create product and seller summary at order level

product_seller_summary = (
    order_items_enriched
    .groupby("order_id")
    .agg(
        main_product_category=("product_category_name_english", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        main_seller_state=("seller_state", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        avg_product_weight_g=("product_weight_g", "mean"),
        avg_product_length_cm=("product_length_cm", "mean"),
        avg_product_height_cm=("product_height_cm", "mean"),
        avg_product_width_cm=("product_width_cm", "mean")
    )
    .reset_index()
)

product_seller_summary.head()

,order_id,main_product_category,main_seller_state,avg_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,SP,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,SP,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,MG,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,perfumery,SP,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,PR,3750.0,35.0,40.0,30.0


In [20]:
# Create the analysis-ready master dataset

master_orders = (
    delivered_orders
    .merge(customers_clean, on="customer_id", how="left")
    .merge(order_items_summary, on="order_id", how="left")
    .merge(payments_summary, on="order_id", how="left")
    .merge(reviews_summary, on="order_id", how="left")
    .merge(product_seller_summary, on="order_id", how="left")
)

master_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,...,total_payment_value,max_payment_installments,avg_review_score,review_count,main_product_category,main_seller_state,avg_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,15,...,38.71,1.0,4.0,1.0,housewares,SP,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,19,...,141.46,1.0,4.0,1.0,perfumery,SP,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,26,...,179.12,3.0,5.0,1.0,auto,SP,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,26,...,72.20,1.0,5.0,1.0,pet_shop,MG,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,12,...,28.62,1.0,5.0,1.0,stationery,SP,250.0,51.0,15.0,15.0


In [21]:
# Check the size of the master dataset

print("Delivered orders rows:", len(delivered_orders))
print("Master dataset rows:", len(master_orders))
print("Master dataset columns:", master_orders.shape[1])

Delivered orders rows: 96478
Master dataset rows: 96478
Master dataset columns: 32


In [22]:
# Check missing values in the master dataset

master_missing = (
    master_orders
    .isna()
    .sum()
    .sort_values(ascending=False)
)

master_missing[master_missing > 0]

main_product_category            1351
avg_review_score                  646
review_count                      646
avg_product_weight_g               16
avg_product_width_cm               16
avg_product_height_cm              16
avg_product_length_cm              16
order_approved_at                  14
delay_days                          8
order_delivered_customer_date       8
delivery_days                       8
order_delivered_carrier_date        2
total_payment_value                 1
max_payment_installments            1
payment_count                       1
dtype: int64

In [23]:
# Export the analysis-ready master dataset

master_output_file = processed_data_path / "analysis_ready_orders.csv"

master_orders.to_csv(master_output_file, index=False)

print("Analysis-ready dataset saved to:")
print(master_output_file)

Analysis-ready dataset saved to:
..\data\processed\analysis_ready_orders.csv


In [24]:
master_output_file.exists()

True